# AC-MOT Master Cache Replay Sweep

One Colab notebook for many cheap AC-MOT experiments using saved detections.

**Safety policy**

- Search/validate cache first.
- Replay uses cached detections only.
- No YOLO inference in this notebook.
- Replay throughput is **not** live FPS.
- FP32 cache results are for development/tuning, not FP16 final evidence.
- The frozen v10_p4 FP16 result remains the publication reference until a finalist is validated live.

Current frozen reference:

| HOTA | MOTA | IDF1 | IDS | Live FP16 FPS |
|---:|---:|---:|---:|---:|
| 22.856 | 11.607 | 21.963 | 270 | 25.9946 |

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, subprocess, json, time, gzip, hashlib, math, shutil
from pathlib import Path
from datetime import datetime

REPO = Path('/content/ACMOT-Codex-V10Style-Portable')
if REPO.exists():
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'], check=False)
else:
    subprocess.run([
        'git','clone',
        'https://github.com/AhmedCode110/ACMOT-Codex-V10Style-Portable.git',
        str(REPO)
    ], check=True)

os.chdir(REPO)
print('Repo:', REPO)
subprocess.run(['git','rev-parse','HEAD'], check=False)
print('ALLOW_LIVE_RUN = False')
ALLOW_LIVE_RUN = False

In [ ]:
# Minimal replay dependencies only. No YOLO model is loaded or executed.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics==8.3.200',
    'motmetrics',
    'opencv-python-headless',
    'pandas',
    'numpy',
    'tqdm',
    'scipy',
    'lap',
    'pyyaml'
], check=True)

import numpy as np
import pandas as pd
import cv2
import motmetrics as mm
from tqdm.auto import tqdm
from collections import deque, Counter
from dataclasses import dataclass
from types import SimpleNamespace

print('Replay environment ready. YOLO inference: NO')

In [ ]:
# Cache discovery / verification / dry-run. These commands must not run YOLO.
commands = [
    [sys.executable, 'cache_manager_v10_p4.py', 'discover'],
    [sys.executable, 'cache_manager_v10_p4.py', 'verify'],
    [sys.executable, 'cache_manager_v10_p4.py', 'status'],
    [sys.executable, 'experiment.py', '--mode', 'replay', '--dry-run'],
]
for cmd in commands:
    print('\n$', ' '.join(cmd))
    proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout)
print('\nCACHE PREFLIGHT COMPLETE')
print('Expensive GPU inference: NO')

In [ ]:
# Locate the best usable legacy multi-resolution detection cache.
MYDRIVE = Path('/content/drive/MyDrive')
KNOWN_CACHE_NAMES = [
    'detection_cache_v10_p3_yolov8n_fp32_640_736_832',
    'detection_cache_v1',
]

def quick_cache_score(p: Path):
    if not p.is_dir():
        return -1
    seq_files = list(p.glob('*.jsonl.gz'))
    done_files = list(p.glob('*.complete.json'))
    meta = p / 'cache_meta.json'
    score = len(seq_files) * 10 + len(done_files) * 3 + (50 if meta.exists() else 0)
    if '640_736_832' in p.name:
        score += 100
    return score

candidate_paths = []
expected = MYDRIVE / 'VisDrone_Results' / 'ACMOT_CODEX_V10STYLE' / 'detection_cache_v10_p3_yolov8n_fp32_640_736_832'
if expected.exists():
    candidate_paths.append(expected)

# Limited recursive name search under the most likely project roots.
search_roots = [
    MYDRIVE / 'VisDrone_Results',
    MYDRIVE / 'ACMOT_IDS',
]
for root in search_roots:
    if root.exists():
        for name in KNOWN_CACHE_NAMES:
            try:
                candidate_paths.extend([p for p in root.rglob(name) if p.is_dir()])
            except Exception:
                pass

# Deduplicate
unique = []
seen = set()
for p in candidate_paths:
    s = str(p.resolve())
    if s not in seen:
        seen.add(s); unique.append(p)

if not unique:
    raise RuntimeError('No legacy detection cache found. Stop: NO YOLO will be started.')

ranked = sorted([(quick_cache_score(p), p) for p in unique], reverse=True)
for score, p in ranked:
    print(score, p)

CACHE_DIR = ranked[0][1]
print('\nSelected cache:', CACHE_DIR)

meta_path = CACHE_DIR / 'cache_meta.json'
CACHE_META = json.loads(meta_path.read_text()) if meta_path.exists() else {}
print(json.dumps(CACHE_META, indent=2))

PRECISION = str(CACHE_META.get('precision') or ('FP32' if 'fp32' in CACHE_DIR.name.lower() else 'UNKNOWN')).upper()
SIZES = CACHE_META.get('sizes', [640, 736, 832])
print('Precision:', PRECISION)
print('Sizes:', SIZES)
print('Development replay valid: YES')
print('Final FP16 evidence valid:', 'YES' if PRECISION == 'FP16' else 'NO')
print('Live FPS valid: NO')

In [ ]:
# Verify sequence completion receipts and hashes where possible.
cache_files = sorted(CACHE_DIR.glob('*.jsonl.gz'))
if not cache_files:
    raise RuntimeError('Selected cache has no .jsonl.gz sequence files.')

verification = []
for fp in cache_files:
    seq = fp.name[:-9]  # remove .jsonl.gz
    done = CACHE_DIR / f'{seq}.complete.json'
    rec = {'sequence': seq, 'cache_file': str(fp), 'complete_file': str(done), 'status': 'UNKNOWN'}
    if not done.exists():
        rec['status'] = 'PARTIAL'
    else:
        info = json.loads(done.read_text())
        rec['frames'] = int(info.get('frames', 0))
        expected_sha = info.get('sha256')
        if expected_sha:
            h = hashlib.sha256()
            with open(fp, 'rb') as f:
                for block in iter(lambda: f.read(1024*1024), b''):
                    h.update(block)
            rec['sha_match'] = (h.hexdigest() == expected_sha)
            rec['status'] = 'VALID' if rec['sha_match'] else 'CORRUPT'
        else:
            rec['status'] = 'VALID'
    verification.append(rec)

verify_df = pd.DataFrame(verification)
display(verify_df)
bad = verify_df[~verify_df.status.eq('VALID')]
if len(bad):
    raise RuntimeError(f'Cache verification failed for {len(bad)} sequences. No replay started.')
print('Validated sequence caches:', len(verify_df))

In [ ]:
# Locate GT annotations. Replay metrics do not require staging all images.
dataset_candidates = [
    MYDRIVE / 'visdrone' / 'VisDrone_Zips' / 'VisDrone2019-MOT-test-dev' / 'VisDrone2019-MOT-test-dev',
]
DATASET = next((p for p in dataset_candidates if (p/'annotations').exists()), None)
if DATASET is None:
    # fallback limited search
    roots = [MYDRIVE / 'visdrone', MYDRIVE / 'VisDrone_Results']
    found = []
    for root in roots:
        if root.exists():
            try:
                for p in root.rglob('VisDrone2019-MOT-test-dev'):
                    if (p/'annotations').exists():
                        found.append(p)
            except Exception:
                pass
    DATASET = found[0] if found else None

if DATASET is None:
    raise RuntimeError('VisDrone GT dataset not found. Stop before replay metrics.')

ANN_DIR = DATASET / 'annotations'
ann_files = sorted(ANN_DIR.glob('*.txt'))
print('Dataset:', DATASET)
print('Annotation files:', len(ann_files))
if len(ann_files) != 17:
    raise RuntimeError(f'Expected 17 GT annotation files, found {len(ann_files)}')

SEQ_NAMES = [Path(r['cache_file']).name[:-9] for r in verification]
missing_gt = [s for s in SEQ_NAMES if not (ANN_DIR / f'{s}.txt').exists()]
if missing_gt:
    raise RuntimeError('Missing GT for: ' + ', '.join(missing_gt))

print('GT coverage OK for cached sequences.')

In [ ]:
# Core replay logic adapted from the project's existing v10_p3 fast replay.
from ultralytics.trackers.byte_tracker import BYTETracker
from ultralytics.engine.results import Boxes

COCO_CLASSES = [0, 2, 5, 7]
VISDRONE_GT_CLASSES = [1, 4, 5, 6, 9]

@dataclass
class SceneState:
    sci: float = 0.0
    scene: str = 'clear'
    brightness: float = 128.0
    blur: float = 500.0
    edge_density: float = 0.0
    crowd: float = 0.0
    tiny_ratio: float = 0.0
    n_dets: int = 0

class ReplayAnalyzer:
    def __init__(self, cfg):
        self.cfg = cfg
        self.hist = deque(maxlen=int(cfg.get('window', 7)))
        self.state = SceneState()

    def maybe_update(self, frame_id, visual, prev_boxes):
        stride = int(self.cfg.get('stride', 10))
        if frame_id != 1 and (frame_id - 1) % stride != 0:
            return self.state

        brightness = float(visual.get('brightness', 128.0))
        blur = float(visual.get('blur', 500.0))
        edge_density = float(visual.get('edge_density', 0.0))
        n = len(prev_boxes)
        crowd_div = float(self.cfg.get('crowd_div', 30.0))
        crowd = min(n / crowd_div, 1.0)
        if n:
            areas = (prev_boxes[:,2]-prev_boxes[:,0]) * (prev_boxes[:,3]-prev_boxes[:,1])
            tiny_ratio = float(np.mean(areas < float(self.cfg.get('tiny_area', 32*32))))
        else:
            tiny_ratio = 0.0

        edge_norm = min(edge_density / float(self.cfg.get('edge_norm', 0.14)), 1.0)
        raw = (
            float(self.cfg.get('w_crowd', .30))*crowd +
            float(self.cfg.get('w_tiny', .30))*tiny_ratio +
            float(self.cfg.get('w_edge', .20))*edge_norm +
            float(self.cfg.get('w_night', .10))*float(brightness < float(self.cfg.get('night_thr', 80))) +
            float(self.cfg.get('w_blur', .05))*float(blur < float(self.cfg.get('blur_thr', 180)))
        )
        self.hist.append(float(np.clip(raw, 0, 1)))
        sci = float(np.mean(self.hist))

        if brightness < float(self.cfg.get('night_thr', 80)):
            scene = 'night'
        elif blur < float(self.cfg.get('blur_thr', 180)):
            scene = 'blur'
        elif tiny_ratio > float(self.cfg.get('tiny_scene_thr', .50)):
            scene = 'tiny'
        elif crowd > float(self.cfg.get('crowd_scene_thr', .65)) or edge_density > float(self.cfg.get('edge_scene_thr', .13)):
            scene = 'crowded'
        else:
            scene = 'clear'

        self.state = SceneState(sci, scene, brightness, blur, edge_density, crowd, tiny_ratio, n)
        return self.state

def load_gt(path):
    cols = ['frame','id','x','y','w','h','score','cat','trunc','occ']
    df = pd.read_csv(path, header=None, names=cols)
    df = df[df['cat'].isin(VISDRONE_GT_CLASSES)]
    df = df[(df['score']==1) & (df['occ']<2) & (df['trunc']<2)]
    return df.reset_index(drop=True)

def iou_distance(pred_xyxy, gt_xyxy):
    pred = np.asarray(pred_xyxy, float).reshape(-1,4)
    gt = np.asarray(gt_xyxy, float).reshape(-1,4)
    if not len(pred) or not len(gt):
        return np.empty((len(gt), len(pred)))
    ix1 = np.maximum(pred[:,0][None,:], gt[:,0][:,None])
    iy1 = np.maximum(pred[:,1][None,:], gt[:,1][:,None])
    ix2 = np.minimum(pred[:,2][None,:], gt[:,2][:,None])
    iy2 = np.minimum(pred[:,3][None,:], gt[:,3][:,None])
    inter = np.maximum(0,ix2-ix1)*np.maximum(0,iy2-iy1)
    ap = (pred[:,2]-pred[:,0])*(pred[:,3]-pred[:,1])
    ag = (gt[:,2]-gt[:,0])*(gt[:,3]-gt[:,1])
    union = ap[None,:]+ag[:,None]-inter
    iou = np.divide(inter, union, out=np.zeros_like(inter), where=union>0)
    dist = 1.0 - iou
    # MOT matching gate consistent with standard 0.5 IoU association.
    dist[iou < 0.5] = np.nan
    return dist

def make_tracker(tp):
    return BYTETracker(SimpleNamespace(
        track_high_thresh=float(tp['high']),
        track_low_thresh=float(tp['low']),
        new_track_thresh=float(tp['new']),
        track_buffer=int(tp['buffer']),
        match_thresh=float(tp['match']),
        fuse_score=bool(tp.get('fuse', True)),
    ), frame_rate=30)

def calibrate(trial, state):
    c = trial['calib']
    conf = float(c.get('conf_base', .245)) - float(c.get('conf_slope', .050))*state.sci
    if state.scene in {'crowded','tiny','night'}:
        conf -= float(c.get('scene_conf_nudge', .012))
    conf = float(np.clip(conf, float(c.get('conf_floor', .19)), float(c.get('conf_ceil', .28))))

    sci_mid = float(c.get('sci_mid', .35))
    sci_high = float(c.get('sci_high', .60))
    tiny_gate = float(c.get('tiny_gate', .50))
    crowd_gate = c.get('crowd_gate', None)

    if state.sci > sci_high or state.tiny_ratio > tiny_gate or (crowd_gate is not None and state.crowd > float(crowd_gate)):
        imgsz = 832
    elif state.sci > sci_mid or state.scene in {'crowded','tiny'}:
        imgsz = 736
    else:
        imgsz = 640
    return {'conf': conf, 'imgsz': imgsz}

def track_cached(tracker, dets, shape, conf, tp):
    arr = np.asarray(dets, float).reshape(-1,6) if len(dets) else np.empty((0,6), float)
    if len(arr):
        arr = arr[arr[:,4] >= conf]
    tracker.args.track_high_thresh = float(tp['high'])
    tracker.args.track_low_thresh = float(tp['low'])
    tracker.args.new_track_thresh = float(tp['new'])
    tracker.args.match_thresh = float(tp['match'])
    out = np.asarray(tracker.update(Boxes(arr, tuple(shape))), float).reshape(-1,8)
    if len(out):
        return out[:,4].astype(int), out[:,:4], out[:,5]
    return np.array([],int), np.empty((0,4),float), np.array([],float)

def eval_acc(acc):
    mh = mm.metrics.create()
    s = mh.compute(acc, metrics=['mota','idf1','num_switches','recall','precision','num_misses','num_false_positives','num_matches'], name='x')
    r = s.iloc[0]
    tp, fp, fn, ids = int(r['num_matches']), int(r['num_false_positives']), int(r['num_misses']), int(r['num_switches'])
    det_a = tp / max(tp+fp+fn,1)
    ass_a = max(0.0, 1.0 - ids/max(tp,1))
    return dict(
        mota=float(r['mota']), idf1=float(r['idf1']), ids=ids,
        recall=float(r['recall']), precision=float(r['precision']),
        fn=fn, fp=fp, matches=tp,
        hota_approx=float(math.sqrt(max(det_a,0)*max(ass_a,0)))
    )

print('Replay functions ready. YOLO inference: NO')

In [ ]:
# Trial bank: many controlled tries in the SAME notebook.
BASE_TRACKER = dict(high=.18, low=.04, new=.20, buffer=45, match=.86, fuse=True)
BASE_SCI = dict(
    window=7, stride=10,
    w_crowd=.30, w_tiny=.30, w_edge=.20, w_night=.10, w_blur=.05,
    crowd_div=30, edge_norm=.14, night_thr=80, blur_thr=180,
    tiny_area=32*32, tiny_scene_thr=.50, crowd_scene_thr=.65, edge_scene_thr=.13
)
BASE_CALIB = dict(
    conf_base=.245, conf_slope=.050, conf_floor=.19, conf_ceil=.28,
    scene_conf_nudge=.012, sci_mid=.35, sci_high=.60, tiny_gate=.50
)

TRIALS = []

def add_trial(name, group, tracker=None, sci=None, calib=None, notes=''):
    TRIALS.append(dict(
        name=name, group=group,
        tracker={**BASE_TRACKER, **(tracker or {})},
        sci={**BASE_SCI, **(sci or {})},
        calib={**BASE_CALIB, **(calib or {})},
        notes=notes
    ))

# Current reference replay setup
add_trial('ACMOT_CURRENT', 'baseline')

# Tracker-focused
for name, changes in [
    ('T_MATCH_084', {'match':.84}),
    ('T_MATCH_088', {'match':.88}),
    ('T_MATCH_090', {'match':.90}),
    ('T_NEW_022', {'new':.22}),
    ('T_NEW_024', {'new':.24}),
    ('T_BUFFER_60', {'buffer':60}),
    ('T_BUFFER_75', {'buffer':75}),
    ('T_NEW22_MATCH88', {'new':.22,'match':.88}),
    ('T_B60_NEW22_M88', {'buffer':60,'new':.22,'match':.88}),
    ('T_B75_NEW22_M88', {'buffer':75,'new':.22,'match':.88}),
    ('T_HIGH_020', {'high':.20}),
    ('T_LOW_003', {'low':.03}),
    ('T_LOW_005', {'low':.05}),
]:
    add_trial(name, 'tracker', tracker=changes)

# SCI-focused
for name, changes in [
    ('S_TINY_PLUS', {'w_tiny':.35,'w_crowd':.27}),
    ('S_CROWD_PLUS', {'w_crowd':.35,'w_tiny':.27}),
    ('S_EDGE_MINUS', {'w_edge':.15}),
    ('S_WINDOW_5', {'window':5}),
    ('S_WINDOW_9', {'window':9}),
    ('S_STRIDE_5', {'stride':5}),
    ('S_STRIDE_15', {'stride':15}),
    ('S_CROWD_DIV_25', {'crowd_div':25}),
    ('S_CROWD_DIV_35', {'crowd_div':35}),
]:
    add_trial(name, 'sci', sci=changes)

# Calibrator / resolution-focused
for name, changes in [
    ('C_IDS_GUARD', {'conf_base':.248,'conf_slope':.045,'conf_floor':.20,'scene_conf_nudge':.008,'sci_mid':.40,'sci_high':.65}),
    ('C_CONF_FLOOR_020', {'conf_floor':.20}),
    ('C_CONF_FLOOR_021', {'conf_floor':.21}),
    ('C_SLOPE_040', {'conf_slope':.040}),
    ('C_SLOPE_045', {'conf_slope':.045}),
    ('C_MID_040', {'sci_mid':.40}),
    ('C_HIGH_065', {'sci_high':.65}),
    ('C_MID40_HIGH65', {'sci_mid':.40,'sci_high':.65}),
    ('C_DENSITY_GATE_060', {'crowd_gate':.60}),
    ('C_DENSITY_GATE_070', {'crowd_gate':.70}),
    ('C_TINY_GATE_040', {'tiny_gate':.40}),
    ('C_TINY_GATE_060', {'tiny_gate':.60}),
]:
    add_trial(name, 'calibrator', calib=changes)

# Combined promising configurations
add_trial('X_IDS_GUARD_M88', 'combined', tracker={'new':.22,'match':.88}, calib={'conf_base':.248,'conf_slope':.045,'conf_floor':.20,'scene_conf_nudge':.008,'sci_mid':.40,'sci_high':.65})
add_trial('X_IDS_GUARD_B60_M88', 'combined', tracker={'buffer':60,'new':.22,'match':.88}, calib={'conf_base':.248,'conf_slope':.045,'conf_floor':.20,'scene_conf_nudge':.008,'sci_mid':.40,'sci_high':.65})
add_trial('X_DENSITY_B60_M88', 'combined', tracker={'buffer':60,'new':.22,'match':.88}, calib={'crowd_gate':.60,'conf_floor':.20})
add_trial('X_TINYPLUS_IDS', 'combined', tracker={'new':.22,'match':.88}, sci={'w_tiny':.35,'w_crowd':.27}, calib={'conf_floor':.20,'sci_mid':.40,'sci_high':.65})

print('Total trials:', len(TRIALS))
pd.DataFrame([{
    'name':t['name'], 'group':t['group'],
    'match':t['tracker']['match'], 'buffer':t['tracker']['buffer'],
    'new':t['tracker']['new'], 'window':t['sci']['window'],
    'stride':t['sci']['stride'], 'conf_floor':t['calib']['conf_floor'],
    'sci_mid':t['calib']['sci_mid'], 'sci_high':t['calib']['sci_high']
} for t in TRIALS])

In [ ]:
# Sweep dry-run: no tracking yet, no TrackEval, no YOLO.
EXPECTED_SIZES = {640,736,832}
available_sizes = set(int(x) for x in CACHE_META.get('sizes', [640,736,832]))
missing_sizes = EXPECTED_SIZES - available_sizes

print('MASTER SWEEP DRY RUN')
print('Trials:', len(TRIALS))
print('Cache:', CACHE_DIR)
print('Precision:', PRECISION)
print('Cached sizes:', sorted(available_sizes))
print('Missing required sizes:', sorted(missing_sizes))
print('GT:', ANN_DIR)
print('YOLO inference required: NO')
print('Live FPS measured: NO')
print('Replay metrics generated after next cell: YES')

if missing_sizes:
    raise RuntimeError('Required multi-resolution cache is incomplete. Stop before sweep.')

In [ ]:
# Run all replay trials. This is CPU/cache work; it does NOT execute YOLO.
RUN_TAG = 'master_sweep_' + datetime.now().strftime('%Y%m%d_%H%M%S')
SWEEP_ROOT = MYDRIVE / 'VisDrone_Results' / 'ACMOT_SWEEPS' / RUN_TAG
SWEEP_ROOT.mkdir(parents=True, exist_ok=False)
TRIAL_ROOT = SWEEP_ROOT / 'trial_outputs'
TRIAL_ROOT.mkdir(parents=True, exist_ok=True)

def run_trial(trial):
    rows = []
    pred_root = TRIAL_ROOT / trial['name'] / 'predictions_mot'
    pred_root.mkdir(parents=True, exist_ok=True)
    total_frames = sum(int(x.get('frames') or 0) for x in verification)
    pbar = tqdm(total=total_frames, desc=trial['name'], dynamic_ncols=True)
    for seq in SEQ_NAMES:
        gt = load_gt(ANN_DIR / f'{seq}.txt')
        tracker = make_tracker(trial['tracker'])
        analyzer = ReplayAnalyzer(trial['sci'])
        prev_boxes = np.empty((0,4),float)
        acc = mm.MOTAccumulator(auto_id=True)
        pred_lines = []
        imgsz_log, conf_log, replay_times = [], [], []
        cache_file = CACHE_DIR / f'{seq}.jsonl.gz'
        with gzip.open(cache_file, 'rt', encoding='utf-8') as f:
            for line in f:
                t0 = time.perf_counter()
                rec = json.loads(line)
                frame_id = int(rec['frame'])
                state = analyzer.maybe_update(frame_id, rec.get('visual',{}), prev_boxes)
                params = calibrate(trial, state)
                bank = rec['bank']
                key = str(params['imgsz'])
                if key not in bank:
                    raise RuntimeError(f"{trial['name']}: missing imgsz={key} in {seq} frame {frame_id}")
                ids, boxes, scores = track_cached(tracker, bank[key], rec['shape'], params['conf'], trial['tracker'])
                prev_boxes = boxes.copy()

                for tid, box, score in zip(ids, boxes, scores):
                    x1,y1,x2,y2 = [float(v) for v in box]
                    pred_lines.append(f'{frame_id},{int(tid)},{x1:.2f},{y1:.2f},{x2-x1:.2f},{y2-y1:.2f},{float(score):.6f},-1,-1,-1\n')

                gt_f = gt[gt.frame == frame_id]
                gt_ids = gt_f.id.values
                gt_boxes = np.column_stack([gt_f.x,gt_f.y,gt_f.x+gt_f.w,gt_f.y+gt_f.h]) if len(gt_f) else np.empty((0,4))
                dist = iou_distance(boxes, gt_boxes)
                acc.update(gt_ids, ids, dist)

                imgsz_log.append(params['imgsz'])
                conf_log.append(params['conf'])
                replay_times.append(time.perf_counter()-t0)
                pbar.update(1)

        (pred_root / f'{seq}.txt').write_text(''.join(pred_lines), encoding='utf-8')
        m = eval_acc(acc)
        rows.append(dict(
            trial=trial['name'], group=trial['group'], sequence=seq,
            frames=len(imgsz_log),
            replay_fps_not_live=len(imgsz_log)/max(sum(replay_times),1e-9),
            mean_imgsz=float(np.mean(imgsz_log)) if imgsz_log else 0,
            mean_conf=float(np.mean(conf_log)) if conf_log else 0,
            **m
        ))
    pbar.close()

    out_dir = TRIAL_ROOT / trial['name']
    pd.DataFrame(rows).to_csv(out_dir/'per_sequence_metrics.csv', index=False)
    (out_dir/'trial_config.json').write_text(json.dumps(trial, indent=2), encoding='utf-8')
    return rows

all_rows = []
for i, trial in enumerate(TRIALS, 1):
    print(f'\n[{i}/{len(TRIALS)}] {trial["name"]}')
    all_rows.extend(run_trial(trial))

per_seq = pd.DataFrame(all_rows)
per_seq.to_csv(SWEEP_ROOT/'SWEEP_PER_SEQUENCE.csv', index=False)
print('Saved:', SWEEP_ROOT)

In [ ]:
# Aggregate leaderboard. HOTA here is an approximation until official TrackEval is run on finalists.
summary = []
for trial_name, g in per_seq.groupby('trial', sort=False):
    summary.append(dict(
        trial=trial_name,
        group=g['group'].iloc[0],
        mota=g['mota'].mean()*100,
        idf1=g['idf1'].mean()*100,
        ids=int(g['ids'].sum()),
        recall=g['recall'].mean()*100,
        precision=g['precision'].mean()*100,
        hota_approx=g['hota_approx'].mean()*100,
        mean_imgsz=g['mean_imgsz'].mean(),
        replay_fps_not_live=g['replay_fps_not_live'].mean(),
    ))
leader = pd.DataFrame(summary)

# Frozen official reference is separate from replay values.
REF = dict(hota=22.856, mota=11.607, idf1=21.963, ids=270)
leader['delta_mota_vs_frozen'] = leader.mota - REF['mota']
leader['delta_idf1_vs_frozen'] = leader.idf1 - REF['idf1']
leader['delta_ids_vs_frozen'] = leader.ids - REF['ids']

# Balanced normalization inside this compatible sweep only.
def norm_hi(s):
    lo, hi = float(s.min()), float(s.max())
    return (s-lo)/(hi-lo) if hi>lo else pd.Series(np.ones(len(s)), index=s.index)
def norm_lo(s):
    return 1.0 - norm_hi(s)

leader['balanced_score'] = (
    .40*norm_hi(leader.hota_approx) +
    .35*norm_hi(leader.idf1) +
    .15*norm_hi(leader.mota) +
    .10*norm_lo(leader.ids)
)
leader = leader.sort_values(['balanced_score','ids'], ascending=[False,True]).reset_index(drop=True)
leader.insert(0,'rank',np.arange(1,len(leader)+1))

leader['status'] = 'REJECT'
leader.loc[(leader.idf1 >= REF['idf1']) & (leader.ids <= REF['ids']), 'status'] = 'KEEP'
leader.loc[(leader.idf1 > REF['idf1']) & (leader.mota >= REF['mota']) & (leader.ids < REF['ids']), 'status'] = 'LIVE_CANDIDATE'

leader.to_csv(SWEEP_ROOT/'LEADERBOARD.csv', index=False)
leader.to_csv(SWEEP_ROOT/'SWEEP_RESULTS.csv', index=False)
display(leader.head(20))

print('\nBest balanced:')
display(leader.head(5))
print('\nLowest IDS with IDF1 >= frozen reference:')
display(leader[leader.idf1 >= REF['idf1']].sort_values(['ids','idf1'], ascending=[True,False]).head(10))

In [ ]:
# Compare every block of 5 ranked candidates and identify the winner in each block.
blocks = []
for start in range(0, len(leader), 5):
    block = leader.iloc[start:start+5].copy()
    if len(block) == 0:
        continue
    winner = block.sort_values(['balanced_score','ids'], ascending=[False,True]).iloc[0]
    blocks.append({
        'block': f'{start+1}-{start+len(block)}',
        'winner': winner['trial'],
        'balanced_score': winner['balanced_score'],
        'hota_approx': winner['hota_approx'],
        'mota': winner['mota'],
        'idf1': winner['idf1'],
        'ids': int(winner['ids'])
    })
block_df = pd.DataFrame(blocks)
block_df.to_csv(SWEEP_ROOT/'EVERY_5_COMPARISON.csv', index=False)
display(block_df)

In [ ]:
# Pareto front for high IDF1 / low IDS / high MOTA.
def pareto_front(df):
    keep=[]
    rows=df.reset_index(drop=True)
    for i,r in rows.iterrows():
        dominated=False
        for j,q in rows.iterrows():
            if i==j: continue
            better_or_equal = (q.idf1 >= r.idf1) and (q.mota >= r.mota) and (q.ids <= r.ids)
            strictly_better = (q.idf1 > r.idf1) or (q.mota > r.mota) or (q.ids < r.ids)
            if better_or_equal and strictly_better:
                dominated=True; break
        if not dominated:
            keep.append(i)
    return rows.loc[keep].sort_values(['idf1','ids'], ascending=[False,True])

pareto = pareto_front(leader)
pareto.to_csv(SWEEP_ROOT/'PARETO_FRONT.csv', index=False)
display(pareto)

In [ ]:
# Plots
import matplotlib.pyplot as plt
PLOTS = SWEEP_ROOT / 'plots'
PLOTS.mkdir(exist_ok=True)

def save_bar(col, ylabel, filename, n=20):
    d = leader.head(n)
    fig, ax = plt.subplots(figsize=(14,6))
    ax.bar(d.trial, d[col])
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Trial')
    ax.tick_params(axis='x', rotation=75)
    fig.tight_layout()
    fig.savefig(PLOTS/filename, dpi=160)
    plt.show()

save_bar('idf1','IDF1 (%)','idf1_top20.png')
save_bar('mota','MOTA (%)','mota_top20.png')
save_bar('ids','ID switches','ids_top20.png')

fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(leader.ids, leader.idf1)
for _,r in leader.head(10).iterrows():
    ax.annotate(r.trial, (r.ids, r.idf1), fontsize=8)
ax.set_xlabel('ID switches (lower better)')
ax.set_ylabel('IDF1 (%)')
fig.tight_layout()
fig.savefig(PLOTS/'idf1_vs_ids.png', dpi=160)
plt.show()

In [ ]:
# Finalists: no live benchmark is executed here.
finalists = leader[
    (leader.status == 'LIVE_CANDIDATE')
].head(5).copy()

if len(finalists) == 0:
    finalists = leader.head(5).copy()

finalists.to_csv(SWEEP_ROOT/'FINALISTS.csv', index=False)
display(finalists)

print('IMPORTANT')
print('- These are replay candidates.')
print('- hota_approx is NOT official HOTA.')
print('- replay_fps_not_live is NOT live FPS.')
print('- Final FP16 live validation is still required before replacing the frozen result.')
print('- This notebook will NOT launch YOLO because ALLOW_LIVE_RUN =', ALLOW_LIVE_RUN)

In [ ]:
# Save a concise Markdown report.
report = []
report.append('# AC-MOT Master Replay Sweep Report')
report.append('')
report.append(f'- Run: `{RUN_TAG}`')
report.append(f'- Cache: `{CACHE_DIR}`')
report.append(f'- Cache precision: `{PRECISION}`')
report.append(f'- Trials: {len(TRIALS)}')
report.append('- YOLO inference: **NO**')
report.append('- Live FPS measured: **NO**')
report.append('')
report.append('## Frozen official reference')
report.append('')
report.append('| HOTA | MOTA | IDF1 | IDS | Live FP16 FPS |')
report.append('|---:|---:|---:|---:|---:|')
report.append('| 22.856 | 11.607 | 21.963 | 270 | 25.9946 |')
report.append('')
report.append('## Top replay candidates')
report.append('')
report.append(leader.head(10).to_markdown(index=False))
report.append('')
report.append('## Finalists')
report.append('')
report.append(finalists.to_markdown(index=False))
report.append('')
report.append('## Caveats')
report.append('')
report.append('- `hota_approx` is only a development proxy until official TrackEval.')
report.append('- Replay throughput is not deployment FPS.')
report.append('- FP32 replay cache cannot be cited as FP16 evidence.')
report.append('- Finalists require a separate live FP16 validation before publication.')

(SWEEP_ROOT/'MASTER_SWEEP_REPORT.md').write_text('\n'.join(report), encoding='utf-8')
print('All outputs saved to:', SWEEP_ROOT)
print('Open LEADERBOARD.csv and MASTER_SWEEP_REPORT.md first.')